# Running `tissue` from Python

`tissue` is a vertex model: cells are polygons, the polygon edges are
walls, and reactions push the vertices around. To run one you need three
files — an **init** (the tissue), a **model** (the reactions) and a
**solver** (how long, how often to print).

Writing those by hand is the hard part, and not because they are long.
The init states its topology explicitly: every wall names the two cells
it separates and the two vertices it joins, and four blocks have to agree
on how many of everything there is. Get a wall's cells wrong and the file
still loads and still runs — it is just a different tissue from the one
you meant.

So none of that is written here. `tissue` derives it from the cell
outlines, which is the only thing you actually have an opinion about.

## 1. A tissue is a list of polygons

In [ ]:
from tissue import TissueInit

t = TissueInit()
t.add_cell([(0, 0), (10, 0), (10, 10), (0, 10)])
t.add_cell([(10, 0), (20, 0), (20, 10), (10, 10)])
t

Two squares side by side. Look at what was derived:

```
(v1, v2, cell1, cell2)      -1 means background
```

In [ ]:
for w in t.walls():
    print(w)

Seven walls, not eight. The edge down the middle is **one** wall with a
cell on each side, because the two outlines gave the same two corner
positions and those were recognised as the same vertices.

That is not bookkeeping. A shared wall is what makes one cell's lobe its
neighbour's indentation — the cells are mechanically the same object
there. Two walls back to back would leave them independent, and nothing
would interdigitate.

## 2. Cells need variables, and wall variable 0 is special

Reactions read and write numbered variables on walls and cells. The one
thing to know: **wall variable 0 is always the resting length** and is
written for you, so the names you give start at index 1.

In [ ]:
t.set_wall_variables(['m', 'force'])          # these are walls 1 and 2
t.set_cell_variables(['cellType', 'A0'],      # cells 0 and 1
                     defaults={'A0': 'area'}) # seed A0 from the polygon
print(t.to_text())

## 3. A model is a list of reactions

You do not have to know a reaction's parameter count — ask.

In [ ]:
from tissue import reactions, describe

print(len(reactions()), 'reactions available')
print(reactions('AreaElastic'))
describe('Pressure2D::AreaElastic')

`parameters` names them in order and `index_levels` says how many indices
each level takes. Some reactions check their arguments in the constructor
instead, and then the error message is the documentation:

In [ ]:
for line in describe('WallMechanics::Spring')['usage']:
    print(line)

Now build the model. Turgor pushes each cell outwards against its resting
area; the walls resist as elastic springs.

In [ ]:
from tissue import Model

m = (Model('turgor against elastic walls')
     .add('Pressure2D::AreaElastic', [3.0, 30.0, 0.0], [1])   # A0 is cell var 1
     .add('WallMechanics::Spring',   [600.0, 1.0],     [0]))  # length is wall var 0
print(m.to_text())

## 4. Run it

`simulate` writes the three files somewhere temporary, runs the simulator
and parses what it prints.

In [ ]:
from tissue import hex_grid, simulate, outlines

# a honeycomb is a better starting tissue than two squares: every
# interior junction joins three cells, as in a real epidermis
tissue = hex_grid(rows=4, cols=4, radius=10.0).subdivide(8)
tissue.set_cell_variables(['cellType', 'A0'], defaults={'A0': 'area'})
print(tissue)

frames = simulate(tissue, m, end=4.0, prints=4)
print(len(frames), 'frames')

`subdivide(8)` is worth pausing on. It splits every cell edge into eight
short walls. With one wall per interface the interface is a straight line
and stays straight however it is pulled — the model **cannot** develop a
wavy boundary. With eight, its interior vertices are free to move. If a
model refuses to make shapes, this is the first thing to check.

## 5. Look at it

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(frames), figsize=(3.2 * len(frames), 3.4))
for ax, fr in zip(axes, frames):
    for _, P in outlines(fr, interior_only=False):
        Q = list(P) + [P[0]]
        ax.fill([p[0] for p in Q], [p[1] for p in Q],
                facecolor='#cfe0f0', edgecolor='#2f5d8a', lw=1.2)
    ax.set_aspect('equal')
    ax.axis('off')
fig.tight_layout()

## 6. Reading the numbers back

A frame holds the vertices, each cell's vertex indices and variables, and
each wall's endpoints and variables. One trap worth knowing if you ever
parse the raw output yourself: a cell row in the file starts with its
vertex **count**, then that many vertex indices, and only then the
variables — and the declared column count is two more than the number of
variables, because the last two fields are volume and wall count. Reading
either wrong shifts every variable by a constant and the run still scores.
`read_frames` handles both.

In [ ]:
import numpy as np
from shapes import circularity, solidity

for i, fr in enumerate(frames):
    P = [o for _, o in outlines(fr)]
    print(f'frame {i}: {len(P)} interior cells, '
          f'circularity {np.median([circularity(p) for p in P]):.3f}, '
          f'solidity {np.median([solidity(p) for p in P]):.3f}')

## 7. When it goes wrong

The simulator's errors are good — it names the parameters it wanted.

In [ ]:
try:
    simulate(tissue, Model().add('Pressure2D::AreaElastic', [3.0], [1]),
             end=0.1)
except RuntimeError as e:
    print(e)

## 8. Making it lobe

Turgor and springs alone give rounded cells. Lobing needs walls that grow
and that grow *unevenly* — something has to mark where the wall should be
held back. In this model that is a reinforcement variable on each wall,
recruited by curvature, which stiffens the wall and slows its extension.

Three things below are not free choices, and each has cost this project
real time:

- **`k_growth` against `g_tissue`.** Lobe amplitude is set by the ratio of
  wall synthesis to areal growth, not by either alone. Too little wall and
  the cells inflate into circles at the right area.
- **`B_bend`.** Lobe *number* goes as `B^(-1/4)`. A stiffer wall gives
  fewer, longer lobes, and at a high enough value the count is pinned
  before any patterning acts.
- **The cue must be near zero on a straight wall.** Growth goes as
  `1/(1 + gamma*m)`, so a cue with a baseline throttles every wall at once
  and flattens the difference between a neck and a lobe, which is the
  whole of what makes a lobe.

In [ ]:
lobing = (Model('turgor, growth, and curvature-recruited reinforcement')
          .add('Pressure2D::AreaElastic', [3.0, 30.0, 0.0134], [2])
          .add('WallMechanics::SpringModulated', [600.0, 1.0, 4.0],
               [[0], [1], [2]])
          .add('WallMechanics::Bending', [0.2], [[4]])
          .add('CMT::CurvatureRecruitment', [1.0, 0.2, 0.15, 4.0, 0.0],
               [[1]])
          .add('WallGrowth::StrainWallInhibited',
               [0.048, -1.0, 9.0, 0.0, 0.0], [[0], [1]])
          .add('Diagnostic::WallCurvature', [], [[3]])
          .add('GrowthForce::BoundaryDilation', [0.0067], []))

leaf = hex_grid(5, 5, radius=10.0).subdivide(12)
leaf.set_wall_variables(['m', 'force', 'curvature', 'bendFlag'],
                        defaults={'bendFlag': 1.0})
leaf.set_cell_variables(['cellType', 'hexRing', 'A0'],
                        defaults={'A0': 'area'})
print(leaf)

This one takes a few minutes. Reduce `end` to try it quickly.

In [ ]:
lobed = simulate(leaf, lobing, end=24.0, prints=6)

fig, axes = plt.subplots(1, len(lobed), figsize=(2.8 * len(lobed), 3.0))
for ax, fr in zip(axes, lobed):
    for _, P in outlines(fr, interior_only=False):
        Q = list(P) + [P[0]]
        ax.fill([p[0] for p in Q], [p[1] for p in Q],
                facecolor='#e9dff0', edgecolor='#7a3f8f', lw=1.0)
    ax.set_aspect('equal'); ax.axis('off')
fig.tight_layout()

## Where to go next

- `tissue.reactions()` lists all 206 of them; `describe(name)` says
  what each wants.
- `write_all(tissue, model, end=..., directory='somewhere')` writes the
  three files without running, if you would rather drive the simulator
  yourself.
- `Frame.interior()` drops the boundary ring, whose shape is set by the
  edge of the patch rather than by the model — most measurements want that.
- Score runs **at matched shape**, not at matched time. Two runs reach
  different shapes in the same simulated time, so comparing their endpoints
  compares different shapes.